In [1]:
# Function 6 - BBO Capstone Project
# Strategy: Scaled Gaussian Process + EI/UCB Hybrid
# Function 6 is a 5D black-box optimization problem.

import numpy as np
from scipy.stats import norm, qmc
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


# --------------------------------------------------
# 1. Function 6 data
# --------------------------------------------------

X = np.array([
    [0.7281861,  0.15469257, 0.73255167, 0.69399651, 0.05640131],
    [0.24238435, 0.84409997, 0.5778091,  0.67902128, 0.50195289],
    [0.72952261, 0.7481062,  0.67977464, 0.35655228, 0.67105368],
    [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
    [0.6188123,  0.33180214, 0.18728787, 0.75623847, 0.3288348 ],
    [0.78495809, 0.91068235, 0.7081201,  0.95922543, 0.0049115 ],
    [0.14511079, 0.8966846,  0.89632223, 0.72627154, 0.23627199],
    [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
    [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
    [0.75759436, 0.35583141, 0.0165229,  0.4342072,  0.11243304],
    [0.5367969,  0.30878091, 0.41187929, 0.38822518, 0.5225283 ],
    [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
    [0.6293079,  0.80348368, 0.81140844, 0.04561319, 0.11062446],
    [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
    [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
    [0.25890557, 0.79367771, 0.6421139,  0.19667346, 0.59310318],
    [0.43216593, 0.71561781, 0.3418191,  0.70499988, 0.61496184],
    [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
    [0.9217762,  0.93187122, 0.41487637, 0.59505727, 0.73562569],
    [0.12667892, 0.2914703,  0.06452848, 0.6805146,  0.89281919],

    # Iteration results from our project
    [0.125720, 0.862724, 0.028544, 0.246605, 0.751206],
    [0.911297, 0.360128, 0.681187, 0.797057, 0.681155],
    [0.7281861,  0.15469257, 0.73255167, 0.69399651, 0.05640131],
    [0.362369, 0.752276, 0.832593, 0.669698, 0.158606],
    [0.739844, 0.696416, 0.072917, 0.314411, 0.872163],
    [0.885074, 0.601757, 0.716438, 0.837507, 0.396826],
    [0.255160, 0.522740, 0.486545, 0.740267, 0.020766],
    [0.609360, 0.483573, 0.500662, 0.574288, 0.237473],
    [0.393399, 0.196651, 0.599021, 0.857586, 0.016051],
    [0.464984, 0.369388, 0.703268, 0.908164, 0.181040],
    [0.382857, 0.287444, 0.857344, 0.949828, 0.005388],
    [0.425242, 0.406075, 0.602892, 0.950799, 0.394581]
])

y = np.array([
    -0.71426495,
    -1.20995524,
    -1.67219994,
    -1.53605771,
    -0.82923655,
    -1.24704893,
    -1.23378638,
    -1.69434344,
    -2.57116963,
    -1.30911635,
    -1.14478485,
    -1.91267714,
    -1.62283895,
    -1.35668211,
    -2.01842540,
    -1.70255784,
    -1.29424696,
    -0.93575656,
    -2.15576776,
    -1.74688209,

    # Iteration outputs from our project
    -2.61310899866408,
    -1.13914247122897,
    -0.59258464831999,
    -0.828662161603612,
    -2.382478286058939,
    -1.1599256318790203,
    -0.670451143998971,
    -0.7522686466299074,
    -0.44168991392979573,
    -0.2890095251948754,
    -0.5983826226943812,
    -0.5238375129764865
])


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def format_query(point, digits=6):
    """Format the query for portal submission."""
    return "-".join(f"{value:.{digits}f}" for value in point)


def clip_to_bounds(points):
    """Keep candidate values inside [0, 1)."""
    return np.clip(points, 0.0, 0.999999)


def expected_improvement(candidates_scaled, model, scaler_y, y_best, xi=0.01):
    """Expected Improvement in original y scale."""
    mean_scaled, std_scaled = model.predict(candidates_scaled, return_std=True)

    mean = scaler_y.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
    std = std_scaled * scaler_y.scale_[0]
    std = np.maximum(std, 1e-12)

    improvement = mean - y_best - xi
    z = improvement / std

    ei = improvement * norm.cdf(z) + std * norm.pdf(z)
    return ei


def upper_confidence_bound(candidates_scaled, model, scaler_y, kappa=2.0):
    """Upper Confidence Bound in original y scale."""
    mean_scaled, std_scaled = model.predict(candidates_scaled, return_std=True)

    mean = scaler_y.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
    std = std_scaled * scaler_y.scale_[0]

    return mean + kappa * std


# --------------------------------------------------
# 3. Check data and current best result
# --------------------------------------------------

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

best_index = np.argmax(y)
best_point = X[best_index]
best_y = y[best_index]

print("\nBest point so far:")
print(format_query(best_point))
print("Best output so far:", best_y)


# --------------------------------------------------
# 4. Scale the data
# --------------------------------------------------

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

print("\nAfter scaling:")
print(f"X_scaled range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")
print(f"y_scaled range: [{y_scaled.min():.3f}, {y_scaled.max():.3f}]")


# --------------------------------------------------
# 5. Gaussian Process model
# --------------------------------------------------
# Function 6 is 5D, so scaling and controlled noise are useful.

kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-2, 1e3))
    * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(0.1, 100.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e-1)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.01,
    normalize_y=False,
    n_restarts_optimizer=25,
    random_state=42
)

gpr.fit(X_scaled, y_scaled)

training_score = gpr.score(X_scaled, y_scaled)

print("\nGP Model Diagnostics:")
print("Kernel:", gpr.kernel_)
print("Training R²:", round(training_score, 4))

if training_score > 0.99:
    print("Warning: Training R² is very high. Check for possible overfitting.")


# --------------------------------------------------
# 6. Generate candidate points
# --------------------------------------------------
# Strategy:
# 50% local exploitation around best point
# 50% global exploration because Function 6 is 5D.

np.random.seed(42)

local_candidates = best_point + np.random.normal(
    loc=0.0,
    scale=0.10,
    size=(8000, X.shape[1])
)

local_candidates = clip_to_bounds(local_candidates)

sampler = qmc.LatinHypercube(d=X.shape[1], seed=42)
global_candidates = sampler.random(n=8000)

X_candidates = np.vstack([
    local_candidates,
    global_candidates
])

X_candidates_scaled = scaler_X.transform(X_candidates)


# --------------------------------------------------
# 7. Acquisition scoring: EI + UCB
# --------------------------------------------------

EI_XI = 0.01
UCB_KAPPA = 2.0

ei_scores = expected_improvement(
    candidates_scaled=X_candidates_scaled,
    model=gpr,
    scaler_y=scaler_y,
    y_best=best_y,
    xi=EI_XI
)

ucb_scores = upper_confidence_bound(
    candidates_scaled=X_candidates_scaled,
    model=gpr,
    scaler_y=scaler_y,
    kappa=UCB_KAPPA
)

# Normalize EI and UCB
ei_norm = (ei_scores - ei_scores.min()) / (ei_scores.max() - ei_scores.min() + 1e-12)
ucb_norm = (ucb_scores - ucb_scores.min()) / (ucb_scores.max() - ucb_scores.min() + 1e-12)

# Hybrid score:
# 60% EI for improvement, 40% UCB for exploration.
hybrid_score = 0.60 * ei_norm + 0.40 * ucb_norm


# --------------------------------------------------
# 8. Select next query
# --------------------------------------------------

best_candidate_index = np.argmax(hybrid_score)
x_next = X_candidates[best_candidate_index]

pred_scaled, std_scaled = gpr.predict(
    scaler_X.transform(x_next.reshape(1, -1)),
    return_std=True
)

predicted_y = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()[0]
predicted_std = std_scaled[0] * scaler_y.scale_[0]

print("\nNext Point to Sample:")
print("X =", x_next)
print("Submission format:", format_query(x_next))
print("Predicted y:", predicted_y)
print("Uncertainty:", predicted_std)
print("Hybrid score:", hybrid_score[best_candidate_index])


# --------------------------------------------------
# 9. Show top 5 candidates
# --------------------------------------------------

top5_index = np.argsort(hybrid_score)[-5:][::-1]

print("\nTop 5 Candidates:")

for rank, idx in enumerate(top5_index, start=1):
    point = X_candidates[idx]

    pred_i_scaled, std_i_scaled = gpr.predict(
        scaler_X.transform(point.reshape(1, -1)),
        return_std=True
    )

    pred_i = scaler_y.inverse_transform(pred_i_scaled.reshape(-1, 1)).ravel()[0]
    std_i = std_i_scaled[0] * scaler_y.scale_[0]

    print(
        f"{rank}. X={format_query(point)} | "
        f"pred={pred_i:.6f}, "
        f"std={std_i:.6f}, "
        f"score={hybrid_score[idx]:.6f}"
    )


# --------------------------------------------------
# 10. Sanity checks
# --------------------------------------------------

print("\nSanity Checks:")
print(f"Training y range: [{y.min():.3f}, {y.max():.3f}]")
print(f"Best observed y: {best_y:.6f}")
print(f"Number of candidate points: {len(X_candidates)}")

Shape of X: (32, 5)
Shape of y: (32,)

Best point so far:
0.464984-0.369388-0.703268-0.908164-0.181040
Best output so far: -0.2890095251948754

After scaling:
X_scaled range: [-2.062, 1.724]
y_scaled range: [-2.128, 1.666]

GP Model Diagnostics:
Kernel: 1.93**2 * Matern(length_scale=[4.28, 5.49, 4.4, 7.03, 9.73], nu=2.5) + WhiteKernel(noise_level=0.0176)
Training R²: 0.9905

Next Point to Sample:
X = [0.48189353 0.38003138 0.6009548  0.999999   0.        ]
Submission format: 0.481894-0.380031-0.600955-0.999999-0.000000
Predicted y: -0.34064172466064135
Uncertainty: 0.118900386423236
Hybrid score: 0.9992920540244907

Top 5 Candidates:
1. X=0.481894-0.380031-0.600955-0.999999-0.000000 | pred=-0.340642, std=0.118900, score=0.999292
2. X=0.517613-0.327217-0.548484-0.997322-0.005045 | pred=-0.347078, std=0.124204, score=0.998013
3. X=0.468825-0.392007-0.618647-0.993233-0.000000 | pred=-0.343455, std=0.116585, score=0.955138
4. X=0.422152-0.368341-0.589851-0.999999-0.001626 | pred=-0.345417,